# 21j — Series 21: **the effect of more runs** (N_RUNS sweep) — no clip, LR step-matched

**Series 21 (supervised). ONE knob: `N_RUNS ∈ {100, 200, 400}`** (×1, ×2, ×4). Everything else is 21i
verbatim — no gradient clip, no parameter clamps, Anuar's SUM log-likelihood reward, score denominator =
`sigma_prop`, divergence guard, 8-exp benchmark, `N_ITER=30`.

**The design subtlety.** The μ REINFORCE gradient scales with the number of draws
(`grad_mu = −(r−r̄)·score`, summed over draws → ∝ N_RUNS), while the γ gradient does **not** (its weights
`w` are row-normalized, so `s_gamma` is M-invariant). So to measure the *noise* effect and not a step-size
effect:

    LR_MU(N_RUNS) = 1e-4 * (100 / N_RUNS)      -> 1e-4, 5e-5, 2.5e-5   (expected step unchanged)
    LR_GAMMA      = 0.5 unchanged

**Prediction (falsifiable).** (a) The μ *landing* stays ≈ the same (the estimator is ~unbiased; 21i bias
+7.3%) but the step-to-step scatter shrinks like 1/√M ⇒ μ rel-RMSE improves below 40.6%.
(b) the 1nW T05 **γ divergence persists** (its gradient is M-invariant, so more runs cannot cure it).

## Panel
1. **FIG 1** μ/mu_true vs transmission, one line per N_RUNS (1nW | 3nW).
2. **FIG 2** μ trajectories (μ/mu_true vs step), one line per N_RUNS, per experiment.
3. **FIG 3** summary vs N_RUNS: μ rel-RMSE, γ rel-RMSE, μ-ratio scatter (std over exps), and trajectory
   roughness (mean |Δ(μ/mu_true)| per step) — the direct noise meter.
4. **FIG 4** (μ,γ) paths for 1nW T05/T20/T60/T100, one path per N_RUNS (init = 0.5×true, origin at (0,0)).
5. **TABLE** per-(N_RUNS, exp) and per-N_RUNS MSE.


## Panel
- **FIG 1** 17h-style `(μ,γ)` phase-space paths (start / step-10 dots / end / truth).
- **FIG 2** per-step gradients: raw `|grad_mu|`, clipped `|grad_mu|`, `|grad_gamma|` (mean ± range per power).
- **FIG 3 ★** recovery ratio `ŷ/true` with Fisher whiskers (μ & γ hue; 1nW top, 3nW bottom; x = transmission).
- **TABLE** final values + MSE/STE.

Convention: figures render **inline only** (no savefig); the notebook is executed **in place**.


In [ ]:
import math, time, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

for _p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    if os.path.isdir(os.path.join(_p, 'src')):
        sys.path.insert(0, _p)
        REPO_ROOT = _p
        break
os.chdir(REPO_ROOT)

from src.fitting import nll, fwhm_from_theta, fit_profile
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK')

In [ ]:
# ============================================================
# EXPERIMENTS — true values from Gregor's fits (identical to 17f/16-series) + real data file
# ============================================================
EXPERIMENTS = [
    dict(name='1nW Trans05',  power='1nW', mu_true=9.393, sigma_prop=2.576, lam=2.232, gamma_true=8.5, n_target=61, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans05.txt'),
    dict(name='1nW Trans10',  power='1nW', mu_true=12.372, sigma_prop=3.445, lam=2.122, gamma_true=8.5, n_target=358, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans10.txt'),
    dict(name='1nW Trans20',  power='1nW', mu_true=17.316, sigma_prop=4.141, lam=2.286, gamma_true=8.5, n_target=1138, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans20.txt'),
    dict(name='1nW Trans40',  power='1nW', mu_true=38.405, sigma_prop=7.198, lam=2.351, gamma_true=8.5, n_target=2428, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans40.txt'),
    dict(name='1nW Trans60',  power='1nW', mu_true=61.374, sigma_prop=9.851, lam=2.593, gamma_true=8.5, n_target=2424, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans60.txt'),
    dict(name='1nW Trans80',  power='1nW', mu_true=79.365, sigma_prop=12.627, lam=2.758, gamma_true=8.5, n_target=2487, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans80.txt'),
    dict(name='1nW Trans100',  power='1nW', mu_true=70.817, sigma_prop=17.221, lam=2.636, gamma_true=8.5, n_target=2455, data_file='data/raw_data/fwhm_1nW_240221/fwhm_1nW_240221SIL_Puppy_hindleg_red1nW_Top20nW_Trans100.txt'),
    dict(name='3nW Trans05',  power='3nW', mu_true=13.204, sigma_prop=3.724, lam=2.186, gamma_true=14.1, n_target=252, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans05.txt'),
    dict(name='3nW Trans10',  power='3nW', mu_true=24.476, sigma_prop=5.639, lam=2.158, gamma_true=14.1, n_target=1572, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans10.txt'),
    dict(name='3nW Trans20',  power='3nW', mu_true=34.279, sigma_prop=8.319, lam=2.264, gamma_true=14.1, n_target=2171, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans20.txt'),
    dict(name='3nW Trans40',  power='3nW', mu_true=84.892, sigma_prop=24.013, lam=2.475, gamma_true=14.1, n_target=3742, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans40.txt'),
    dict(name='3nW Trans60',  power='3nW', mu_true=103.203, sigma_prop=23.95, lam=2.741, gamma_true=14.1, n_target=2541, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans60.txt'),
    dict(name='3nW Trans80',  power='3nW', mu_true=137.537, sigma_prop=32.107, lam=2.911, gamma_true=14.1, n_target=2508, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans80.txt'),
    dict(name='3nW Trans100',  power='3nW', mu_true=175.707, sigma_prop=40.975, lam=3.087, gamma_true=14.1, n_target=2516, data_file='data/raw_data/fwhm_3nW_210221/fwhm_3nW_210221SIL_Puppy_hindleg_red3nW_Top20nW_Trans100.txt'),
]

# ---- QUICK TEST: uncomment one of these slices ----
# EXPERIMENTS = EXPERIMENTS[:3]                                        # first 3
# EXPERIMENTS = [e for e in EXPERIMENTS if e['name'] in
#                ('1nW Trans80', '3nW Trans80', '3nW Trans100')]      # the hard ones

print(f'{len(EXPERIMENTS)} experiments configured')


In [ ]:
# ============================================================
# SERIES 21 — 21j CONFIG: N_RUNS sweep (no clip, step-matched LR_MU)
# ============================================================
N_ITER   = 30
LR_MU_BASE = 1e-4         # LR_MU at N_RUNS=100 (21i value); rescaled by 100/N_RUNS
LR_GAMMA = 0.5
SIGMA_REF = 10.0          # unused (score denominator = sigma_prop)
GAMMA_SCALE = True
H_REF = 1.0
CLIP     = float('inf')   # no gradient clipping
SEED     = 42
LAMBDA_MEAN = 0.0
H_S_MIN = 0.05
RUNS_LIST = [100, 200, 400]

# 8-exp benchmark (1nW/3nW x Trans05/20/60/100)
BENCH = ['1nW Trans05','1nW Trans20','1nW Trans60','1nW Trans100',
         '3nW Trans05','3nW Trans20','3nW Trans60','3nW Trans100']
EXPERIMENTS = [e for e in EXPERIMENTS if e['name'] in BENCH]

# divergence guard (STOP, not clamp)
MU_GUARD    = 1500.0
GAMMA_GUARD = (0.02, 500.0)

REWARD_MODE = 'loglik_sum'   # Anuar's reward (as in 21i)

M_FINAL = 500
FISHER_SEEDS = 1
FISHER_BASE_SEED = 7000

SMOKE = os.environ.get('NB_SMOKE') == '1'
if SMOKE:
    N_ITER, M_FINAL = 5, 50
    RUNS_LIST = [10, 20]
    EXPERIMENTS = EXPERIMENTS[:3]
    print('*** SMOKE RUN ***')

N_CPUS = os.cpu_count() or 4
N_EXP_PARALLEL = 1
N_WORKERS = 4
if N_EXP_PARALLEL * N_WORKERS > N_CPUS:
    N_WORKERS = max(1, N_CPUS // N_EXP_PARALLEL)
print(f'config: N_ITER={N_ITER}, RUNS_LIST={RUNS_LIST}, LR_MU_BASE={LR_MU_BASE}')
print(f'benchmark: {[e["name"] for e in EXPERIMENTS]}')
print(f'parallelism: {N_EXP_PARALLEL} exp(s) x {N_WORKERS} workers on {N_CPUS} CPUs')


In [ ]:
# ============================================================
# helpers: KDE scores (unchanged from 21a) + Anuar's reward
# ============================================================
def _fit_fn(ph):
    return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)
def _fwhm_fn(th):
    return fwhm_from_theta(th, model='lorentzian')
def _nll_fn(th, ph):
    return nll(th, ph, model='lorentzian', uniform_bg=False)

def kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu, sigma_prop):
    d_f = data_f[:, None] - sim_f[None, :]
    d_s = data_s[:, None] - sim_s[None, :]
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * (d_s / h_s) ** 2)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / sigma_prop ** 2
    s_mu = (w * score).sum(dim=1)
    if GAMMA_SCALE:
        dlogG = ((d_f * sim_df[None, :]) / h_f + (d_s * sim_ds[None, :]) / h_s) / H_REF
    else:
        dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    return s_mu, s_gamma, -logp.mean(), w, W

def reward_vector(mode, W):
    '''Anuar's reward: per-run Gaussian log-likelihood of the real data (SUM, no mean).'''
    if mode == 'loglik_sum':
        return torch.log(W.clamp_min(1e-30)).sum(dim=0)
    raise ValueError(mode)

import multiprocessing as _mp
from concurrent.futures import ProcessPoolExecutor as _PPE
def _run_one(args):
    gamma_val, u, b = args
    return compute_fwhm_and_dgamma(gamma_val, u, b, _fit_fn, _fwhm_fn, _nll_fn, n_params=2)
def _init_worker():
    torch.set_num_threads(1)
def _parallel_map(pool, tasks):
    return list(pool.map(_run_one, tasks, chunksize=8))
print('helpers ready')


In [ ]:
# ============================================================
# One experiment (Anuar's reward), NO parameter clamps (21i)
# ============================================================
def load_target(exp):
    d = np.genfromtxt(exp['data_file'])
    fwhm_mhz = d[:, 0] * 1000.0; err_mhz = d[:, 1] * 1000.0
    ok = ~np.isnan(fwhm_mhz) & ~np.isnan(err_mhz) & (fwhm_mhz > 0)
    filt = ok & (err_mhz / fwhm_mhz < 10.0)
    tf = torch.tensor(fwhm_mhz[filt], dtype=torch.float32)
    ts = torch.tensor(err_mhz[filt], dtype=torch.float32)
    n = len(tf); sc = n ** (-1.0 / 6.0)
    hf = float(tf.std()) * sc
    hs = max(float(ts.std()) * sc, H_S_MIN)
    return tf, ts, hf, hs

def run_experiment(exp, n_runs, lr_mu):
    mu_true, sigma_prop = exp['mu_true'], exp['sigma_prop']
    lam, gamma_true = exp['lam'], exp['gamma_true']
    mu_val, gamma_val = 0.5 * mu_true, 0.5 * gamma_true
    target_f, target_s, H_F, H_S = load_target(exp)

    with _PPE(max_workers=N_WORKERS, mp_context=_mp.get_context('fork'),
              initializer=_init_worker) as pool:
        history = []; t0 = time.time()
        diverged = False; diverged_at = None
        for step in range(N_ITER):
            rng2 = np.random.default_rng(SEED + step)
            tasks, ns = [], []
            for _ in range(n_runs):
                u, b, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng2)
                tasks.append((gamma_val, u.numpy(), b.numpy())); ns.append(n)
            res = _parallel_map(pool, tasks)
            ft = torch.tensor([r[0] for r in res], dtype=torch.float32)
            si = torch.tensor([r[1] for r in res], dtype=torch.float32)
            nt = torch.tensor(ns, dtype=torch.float32)
            dg_t = torch.tensor([r[2] for r in res], dtype=torch.float32)
            ds_t = torch.tensor([r[3] for r in res], dtype=torch.float32)

            s_mu, s_gamma, nll_val, w, W = kde_scores(
                ft, si, nt, dg_t, ds_t, target_f, target_s, H_F, H_S, mu_val, sigma_prop)

            r = reward_vector(REWARD_MODE, W)          # Anuar's reward (SUM loglik)
            score = (nt - mu_val) / sigma_prop ** 2    # 21h score denominator
            grad_mu_raw = float(-(r - r.mean()) @ score)       # centered (used)
            grad_mu_raw_u = float(-(r @ score))                # uncentered (diagnostic only)
            grad_mu = float(max(min(grad_mu_raw, CLIP), -CLIP))          # CLIP=inf -> unclipped
            grad_gamma = float(max(min(-s_gamma.mean(), CLIP), -CLIP))   # CLIP=inf -> unclipped

            lr_mu_decay = lr_mu * (1.0 - step / N_ITER)
            mu_val = mu_val - lr_mu_decay * grad_mu                                   # 21i: NO clamp
            gamma_val = gamma_val - LR_GAMMA * (1.0 - 0.5 * step / N_ITER) * grad_gamma  # 21i: NO clamp
            history.append(dict(step=step, mu=mu_val, gamma=gamma_val, nll=float(nll_val),
                                grad_mu=grad_mu, grad_mu_raw=grad_mu_raw,
                                grad_mu_raw_u=grad_mu_raw_u, grad_gamma=grad_gamma,
                                abs_grad_mu=abs(grad_mu), abs_grad_mu_raw=abs(grad_mu_raw),
                                abs_grad_mu_raw_u=abs(grad_mu_raw_u),
                                abs_grad_gamma=abs(grad_gamma), mean_n=float(nt.mean())))
            if (not (0.2 <= mu_val <= MU_GUARD)) or (not (GAMMA_GUARD[0] <= gamma_val <= GAMMA_GUARD[1])):
                diverged = True; diverged_at = step
                break   # left the sane band -> stop (guards compute; NOT a clamp)

        # Fisher / CRB at the fitted point -- skipped if the run diverged (mu/gamma not sane there)
        if diverged:
            std_mu = std_g = float('nan')
        else:
            J_list = []
            for seed in range(FISHER_SEEDS):
                rng = np.random.default_rng(FISHER_BASE_SEED + seed)
                tasks, ns = [], []
                for _ in range(M_FINAL):
                    u, b_, n = draw_fixed_noise(mu_val, sigma_prop, lam, rng)
                    tasks.append((gamma_val, u.numpy(), b_.numpy())); ns.append(n)
                res2 = _parallel_map(pool, tasks)
                ft = torch.tensor([x[0] for x in res2], dtype=torch.float32)
                si2 = torch.tensor([x[1] for x in res2], dtype=torch.float32)
                nn2 = torch.tensor(ns, dtype=torch.float32)
                dg2 = torch.tensor([x[2] for x in res2], dtype=torch.float32)
                ds2 = torch.tensor([x[3] for x in res2], dtype=torch.float32)
                s_mu2, s_g2, _, _, _ = kde_scores(ft, si2, nn2, dg2, ds2,
                                                  target_f, target_s, H_F, H_S, mu_val, sigma_prop)
                s = torch.stack([s_mu2, s_g2], dim=1)
                J_list.append(s.T @ s / len(target_f))
            J = torch.stack(J_list).mean(dim=0)
            Jinv = torch.linalg.inv(J + 1e-8 * torch.eye(2))
            std_mu = math.sqrt(max(Jinv[0, 0].item(), 0.0)); std_g = math.sqrt(max(Jinv[1, 1].item(), 0.0))

    return dict(exp=exp['name'], power=exp['power'], T=int(exp['name'].split('Trans')[1]),
                n_runs=n_runs, lr_mu=lr_mu,
                mu_true=mu_true, gamma_true=gamma_true, sigma_prop=sigma_prop,
                mu_init=0.5 * mu_true, gamma_init=0.5 * gamma_true,
                mu_final=history[-1]['mu'], gamma_final=history[-1]['gamma'], nll_final=history[-1]['nll'],
                std_mu=std_mu, std_gamma=std_g, history=history,
                diverged=diverged, diverged_at=diverged_at)

print('run_experiment ready (Anuar reward, NO parameter clamps)')


In [ ]:
t_all = time.time()
results = []
for n_runs in RUNS_LIST:
    lr_mu = LR_MU_BASE * (100.0 / n_runs)     # keep the expected step constant (grad_mu ~ N_RUNS)
    print(f"--- N_RUNS={n_runs}  (LR_MU={lr_mu:.2e}) ---")
    for exp in EXPERIMENTS:
        r = run_experiment(exp, n_runs, lr_mu)
        results.append(r)
        print(f"{exp['name']:>12}: mu {r['mu_true']:7.2f} -> {r['mu_final']:7.2f} (x{r['mu_final']/r['mu_true']:.2f})"
              f" | gamma {r['gamma_true']:5.1f} -> {r['gamma_final']:9.2f} (x{r['gamma_final']/r['gamma_true']:.2f})"
              f" | diverged={r['diverged']}@{r['diverged_at']}")
    print()
print(f'Total: {(time.time()-t_all)/60:.1f} min')


In [ ]:
# ============================================================
# FIG 1 — mu/mu_true vs transmission, one line per N_RUNS (1nW | 3nW)
# ============================================================
COL = {nr: c for nr, c in zip(RUNS_LIST, ['#1d3557', '#2a9d8f', '#e63946', '#f4a261', '#8a5a44'])}
by = {}
for r in results:
    by.setdefault(r['n_runs'], []).append(r)
EXPS = [e['name'] for e in EXPERIMENTS]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, p_ in zip(axes, ['1nW', '3nW']):
    for nr in RUNS_LIST:
        rs = sorted([r for r in by[nr] if r['power'] == p_], key=lambda z: z['T'])
        ax.plot([r['T'] for r in rs], [r['mu_final'] / r['mu_true'] for r in rs], 'o-', color=COL[nr], label=f'N_RUNS={nr}')
    ax.axhline(1.0, color='k', ls='--', lw=1)
    ax.set_title(f'mu/mu_true — {p_}'); ax.set_xlabel('Transmission (%)'); ax.set_ylabel('mu_hat/mu_true')
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.suptitle('FIG 1 — mu accuracy vs transmission, per N_RUNS'); plt.tight_layout(); plt.show()

# ============================================================
# FIG 2 — mu trajectories (mu/mu_true vs step), lines per N_RUNS
# ============================================================
ncol = 4; nrow = int(math.ceil(len(EXPS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow))
axes = np.asarray(axes).reshape(nrow, ncol)
for k, nm in enumerate(EXPS):
    ax = axes[k // ncol][k % ncol]
    for nr in RUNS_LIST:
        rr = [r for r in by[nr] if r['exp'] == nm][0]
        h = rr['history']
        st = [0] + [x['step'] + 1 for x in h]
        yy = [0.5] + [x['mu'] / rr['mu_true'] for x in h]
        ax.plot(st, yy, '-o', color=COL[nr], ms=3, lw=1.2, label=f'N={nr}')
    ax.axhline(1.0, color='k', ls='--', lw=0.8); ax.set_title(nm, fontsize=9)
    ax.set_xlabel('step'); ax.set_ylabel('mu/mu_true'); ax.grid(alpha=0.3)
    if k == 0: ax.legend(fontsize=6)
for k in range(len(EXPS), nrow * ncol): axes[k // ncol][k % ncol].axis('off')
plt.suptitle('FIG 2 — mu trajectories per experiment (one line per N_RUNS)'); plt.tight_layout(); plt.show()

# ============================================================
# FIG 3 — summary vs N_RUNS: rel-RMSE (mu, gamma), scatter, trajectory roughness
# ============================================================
def roughness(r):
    m = [x['mu'] / r['mu_true'] for x in r['history']]
    return float(np.mean(np.abs(np.diff(m)))) if len(m) > 1 else 0.0
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
xs = RUNS_LIST
mu_rr = []; g_rr = []; scat = []; rough = []; ndiv = []
for nr in xs:
    rs = by[nr]
    rmu = np.array([r['mu_final'] / r['mu_true'] - 1 for r in rs])
    rg = np.array([r['gamma_final'] / r['gamma_true'] - 1 for r in rs])
    mu_rr.append(np.sqrt((rmu ** 2).mean()) * 100)
    g_rr.append(np.sqrt((rg ** 2).mean()) * 100)
    scat.append(np.std([r['mu_final'] / r['mu_true'] for r in rs]))
    rough.append(np.mean([roughness(r) for r in rs]))
    ndiv.append(sum(r['diverged'] for r in rs))
axes[0].plot(xs, mu_rr, 'o-', color='#1d3557'); axes[0].set_title('mu rel-RMSE (%)')
axes[1].plot(xs, g_rr, 'o-', color='#e63946'); axes[1].set_title('gamma rel-RMSE (%)')
axes[2].plot(xs, scat, 'o-', color='#2a9d8f'); axes[2].set_title('mu-ratio scatter (std over exps)')
axes[3].plot(xs, rough, 'o-', color='#f4a261'); axes[3].set_title('trajectory roughness  mean|d(mu/mu_true)|/step')
for ax in axes:
    ax.set_xscale('log'); ax.set_xticks(xs); ax.set_xticklabels([str(v) for v in xs])
    ax.set_xlabel('N_RUNS'); ax.grid(alpha=0.3, which='both')
plt.suptitle('FIG 3 — does more runs reduce noise? (and the diverged-cell counts: %s)' % ndiv)
plt.tight_layout(); plt.show()

# ============================================================
# FIG 4 — (mu,gamma) paths for 1nW T05/T20/T60/T100, one path per N_RUNS
# ============================================================
sel = [nm for nm in ['1nW Trans05', '1nW Trans20', '1nW Trans60', '1nW Trans100'] if nm in EXPS]
fig, axes = plt.subplots(1, len(sel), figsize=(5.5 * len(sel), 5))
for ax, nm in zip(np.asarray(axes).reshape(-1), sel):
    for nr in RUNS_LIST:
        rr = [r for r in by[nr] if r['exp'] == nm][0]
        mu_t, g_t = rr['mu_true'], rr['gamma_true']
        mup = [0.5 * mu_t] + [x['mu'] for x in rr['history']]
        gp = [0.5 * g_t] + [x['gamma'] for x in rr['history']]
        ax.plot(mup, gp, '-o', color=COL[nr], ms=3, lw=1.2, label=f'N={nr}')
    rr = [r for r in by[RUNS_LIST[0]] if r['exp'] == nm][0]
    ax.plot(0.5 * rr['mu_true'], 0.5 * rr['gamma_true'], 'o', mfc='none', mec='k', mew=1.6, ms=11)
    ax.axvline(rr['mu_true'], color='gray', ls='--', lw=0.8); ax.axhline(rr['gamma_true'], color='gray', ls='--', lw=0.8)
    ax.plot(rr['mu_true'], rr['gamma_true'], 'k+', ms=11)
    ax.set_xlim(left=0); ax.set_ylim(bottom=0); ax.set_title(nm); ax.set_xlabel('mu (photons)'); ax.set_ylabel('gamma (MHz)')
    ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.suptitle('FIG 4 — (mu,gamma) paths per N_RUNS (hollow = 0.5x true init, + = truth, origin (0,0))')
plt.tight_layout(); plt.show()

# ============================================================
# TABLE — per N_RUNS MSE / rel-RMSE (+ diverged count)
# ============================================================
def mse(rel): 
    rel = np.asarray(rel, float); return float(np.sqrt((rel ** 2).mean())), float(rel.mean())
print(f"{'N_RUNS':>7} | {'mu rel-RMSE':>11} {'mu bias':>8} {'mu x range':>15} | {'gam rel-RMSE':>12} {'gam bias':>8} | div")
print('-' * 92)
for nr in RUNS_LIST:
    rs = by[nr]
    rm = [r['mu_final'] / r['mu_true'] - 1 for r in rs]
    rgg = [r['gamma_final'] / r['gamma_true'] - 1 for r in rs]
    m1, b1 = mse(rm); 
    if np.isfinite(rgg).all(): m2, b2 = mse(rgg)
    else: m2, b2 = float('nan'), float('nan')
    xr = [r['mu_final'] / r['mu_true'] for r in rs if not r['diverged']]
    print(f"{nr:>7} | {m1*100:>11.1f} {b1*100:>+8.1f} {str(round(min(xr),2))+' - '+str(round(max(xr),2)):>15} | {m2*100:>12.1f} {b2*100:>+8.1f} | {sum(r['diverged'] for r in rs)}")
print()
print(f"{'exp':>12} " + ' '.join([f"{'N='+str(v):>18}" for v in RUNS_LIST]))
print('-' * (14 + 19 * len(RUNS_LIST)))
for nm in EXPS:
    cells = []
    for nr in RUNS_LIST:
        rr = [r for r in by[nr] if r['exp'] == nm][0]
        cells.append(f"{rr['mu_final']/rr['mu_true']:.2f}/{rr['gamma_final']/rr['gamma_true']:.2f}{'D' if rr['diverged'] else ' '}")
    print(f"{nm:>12} " + ' '.join([f"{c:>18}" for c in cells]))


## Verdict — to be filled from the executed outputs above

(placeholder: N_RUNS effect on mu noise/landing + the gamma-divergence question.)
